In [3]:
import sys
import os
from neuron import h, gui

# Load mechanisms
# h.nrn_load_dll("/Users/sandek/Desktop/187610master/arm64/libnrnmech.dylib")

# Move to folder with init_cell.py and SWC
os.chdir("/Users/sandek/kws/CA1Sim")
sys.path.append(os.getcwd())

from kws_init_cell import init_cell

cell = init_cell()

from kws_spines_analysis import count_spines
from kws_branch_selection import filter_by_length

# spine analysis
stats = count_spines(cell)
print(stats)

# branch selection
long_apicals = filter_by_length(cell, min_length=100, region="apical")
print(long_apicals)

import os

{'<specify_cells.CA1_Pyr object at 0x103e83340>.basal5': {'length': 29.816485032195892, 'necks': 0, 'heads': 0}, '<specify_cells.CA1_Pyr object at 0x103e83340>.basal6': {'length': 15.492185540328848, 'necks': 7, 'heads': 7}, '<specify_cells.CA1_Pyr object at 0x103e83340>.basal7': {'length': 94.78055988237598, 'necks': 178, 'heads': 178}, '<specify_cells.CA1_Pyr object at 0x103e83340>.basal8': {'length': 4.835283517708445, 'necks': 12, 'heads': 12}, '<specify_cells.CA1_Pyr object at 0x103e83340>.basal9': {'length': 82.419622194081, 'necks': 164, 'heads': 164}, '<specify_cells.CA1_Pyr object at 0x103e83340>.basal10': {'length': 94.20602910222745, 'necks': 172, 'heads': 172}, '<specify_cells.CA1_Pyr object at 0x103e83340>.basal11': {'length': 3.140343913339536, 'necks': 1, 'heads': 1}, '<specify_cells.CA1_Pyr object at 0x103e83340>.basal12': {'length': 74.78600580062319, 'necks': 123, 'heads': 123}, '<specify_cells.CA1_Pyr object at 0x103e83340>.basal13': {'length': 92.15876898304795, 'ne

In [ ]:
## FINAL ## you got this!!!

import os
import random
import numpy as np
import matplotlib.pyplot as plt

from kws_init_cell import init_cell
from kws_pruner import delete_fraction_from_branch_in_one_third, delete_fraction_of_spines_per_branch
from kws_excitatory_synapse import simulate_and_record

def get_peak_voltage(trace):
    return max(trace) - trace[0]

os.chdir("/Users/sandek/kws/CA1Sim")  

target_branch_suffix = "apical131"
n_trials = 100 # ideally 50!
results_dict = {}

# defining the cell types and their pruning schemes
CELL_TYPES = {
    "WT": {"global_prune": 0.0, "plaque_fraction": 0.0, "select_fraction": 1.0}, # WT ## this will run into problem of selecting a minimum of 1 to delete, so used kws_plaque_final_100trials.ipynb to correct
    "SAAeq": {"global_prune": 0.25, "plaque_fraction": 0.33, "select_fraction": 1.0}, # AD-state
    "SAAred": {"global_prune": 0.25, "plaque_fraction": 0.33, "select_fraction": 0.75}, # synapsing 75% less
    "plaque": {"global_prune": 0.0, "plaque_fraction": 0.5, "select_fraction": 1.0}, # WT + plaque (a case that is not biologically relevant but included to see the effect of just the plaque on excitability)
}

for trial in range(n_trials):
    print(f"\n=== Trial {trial + 1}/{n_trials} ===")

    for i in (1, 2, 3):  # each branch third
        print(f"\n  Targeting branch third {i}")

        random.seed()
        np.random.seed()

        # re-initialize cells fresh for this branch third
        cells = {name: init_cell() for name in CELL_TYPES.keys()}

        # apply global SAA pruning 
        for name in ["SAAeq", "SAAred"]:
            delete_fraction_of_spines_per_branch(cells[name], CELL_TYPES[name]["global_prune"])

        # find target branches
        branches = {}
        for name, cell in cells.items():
            for sec_node in cell.apical:
                if sec_node.sec.name().endswith(target_branch_suffix):
                    branches[name] = sec_node
                    break
            if name not in branches:
                raise ValueError(f"Branch {target_branch_suffix} not found in {name}")

        for n in [20]:
            print(f"    Simulating with {n} spines")

            # initialize results_dict hierarchy
            if n not in results_dict:
                results_dict[n] = {}
            if i not in results_dict[n]:
                results_dict[n][i] = {}
            for key in CELL_TYPES.keys():
                if key not in results_dict[n][i]:
                    results_dict[n][i][key] = {"soma": [], "dend": []}

            # there should be enough spines!
            if any(len(branches[name].spines) < n for name in ["WT", "SAAeq"]):
                print(f"      Skipping n={n}: not enough spines")
                continue

            # apply plaque-third pruning and select spines
            spine_selections = {}
            for name, sec_node in branches.items():
                delete_fraction_from_branch_in_one_third(sec_node, i, CELL_TYPES[name]["plaque_fraction"])
                n_select = max(1, int(n * CELL_TYPES[name]["select_fraction"]))
                spine_selections[name] = random.sample(sec_node.spines, n_select)

            # simulate all cells
            for name, sec_node in branches.items():
                spines = spine_selections[name]
                t, v_soma, v_dend, _ = simulate_and_record(cells[name], sec_node, spines)
                results_dict[n][i][name]["soma"].append(get_peak_voltage(v_soma))
                results_dict[n][i][name]["dend"].append(get_peak_voltage(v_dend))

            # pint averages
            for name in CELL_TYPES.keys():
                print(f"      {name}: dend={np.mean(results_dict[n][i][name]['dend']):.2f}, "
                      f"soma={np.mean(results_dict[n][i][name]['soma']):.2f}")


=== Trial 1/100 ===

  Targeting branch third 1
    Simulating with 20 spines
Branch <specify_cells.CA1_Pyr object at 0x16e39a1d0>.apical131 has 320 spines total.
Third 1 has indices 0 to 106, a total 106 spines
Deleting 1 spines out of 106 in this section.
Deleting spine at index 27
	Deleting neck section
Deletion from this branch third done.

Branch <specify_cells.CA1_Pyr object at 0x3022e7460>.apical131 has 240 spines total.
Third 1 has indices 0 to 80, a total 80 spines
Deleting 26 spines out of 80 in this section.
Deleting spine at index 11
	Deleting neck section
Deleting spine at index 22
	Deleting neck section
Deleting spine at index 31
	Deleting neck section
Deleting spine at index 37
	Deleting neck section
Deleting spine at index 25
	Deleting neck section
Deleting spine at index 4
	Deleting neck section
Deleting spine at index 66
	Deleting neck section
Deleting spine at index 43
	Deleting neck section
Deleting spine at index 45
	Deleting neck section
Deleting spine at index 2

KeyboardInterrupt: 

In [ ]:
## SAVING THE DATA ##

import pandas as pd

rows = []
n = 20  # spine count
for third in [1, 2, 3]:
    for cond in ["WT", "plaque", "SAAeq", "SAAred"]:
        for comp in ["soma", "dend"]:   # both compartments
            vals = results_dict[n][third][cond][comp]
            for v in vals:
                rows.append({
                    "spines": n,
                    "branch_third": third,
                    "condition": cond,
                    "compartment": comp,
                    "peak_voltage": v
                })

df = pd.DataFrame(rows)
df.to_csv("100trials_peak_values.csv", index=False)
print("Saved raw data to 100trials_peak_values.csv")


Saved raw data to 100trials_peak_values.csv
